# Barry Plant External Validation Data Preparation

This notebook prepares current Barry Plant rental listings for external validation of the rental-price model trained on the supplied 2025 Domain dataset.

The validation data are cleaned and enriched using the same feature definitions as the training pipeline. Barry Plant observations are reserved for external validation and are not used for model fitting or hyperparameter tuning.

## 1. Load and inspect Barry Plant listings

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os
import requests

from dotenv import load_dotenv
from sklearn.neighbors import BallTree

plant = pd.read_parquet("../data/landing/barryplant_rentals_2026.parquet")

print("Shape:", plant.shape)

Shape: (751, 13)


In [2]:
print("\nMissing values:")
print(plant.isna().sum())

print("\nDuplicate listing IDs:",
        plant["listing_id"].duplicated().sum())

print("\nZero / invalid values:")
print("weekly_rent <= 0:", (plant["weekly_rent"] <= 0).sum())
print("bedrooms < 0:", (plant["bedrooms"] < 0).sum())
print("bathrooms <= 0:", (plant["bathrooms"] <= 0).sum())
print("carspaces < 0:", (plant["carspaces"] < 0).sum())

print("\nCoordinate issues:")
bad_coord = (
    plant["lat"].isna()
    | plant["lon"].isna()
    | ~plant["lat"].between(-44, -33)
    | ~plant["lon"].between(140, 150)
)
print("Invalid / missing coordinates:", bad_coord.sum())

print("\nProperty types:")
print(plant["property_type"].value_counts(dropna=False))


Missing values:
listing_id       0
address          0
suburb           0
postcode         0
weekly_rent      0
bedrooms         0
bathrooms        0
carspaces        0
lat              1
lon              1
url              0
scraped_date     0
property_type    1
dtype: int64

Duplicate listing IDs: 0

Zero / invalid values:
weekly_rent <= 0: 0
bedrooms < 0: 0
bathrooms <= 0: 1
carspaces < 0: 0

Coordinate issues:
Invalid / missing coordinates: 1

Property types:
property_type
House        434
Townhouse    109
Unit          92
Apartment     84
Other         28
Studio         3
NaN            1
Name: count, dtype: int64


## 2. Clean and align property attributes

Property types are mapped to the categories used by the training data. Listings that cannot be reliably aligned with the residential training population are excluded.

Exclusions include invalid coordinates, non-residential listings and property types that cannot be mapped consistently to the training schema.

In [3]:
plant_clean = plant.copy()

plant_clean["suburb"] = (
    plant_clean["suburb"].astype(str).str.strip().str.upper()
)

plant_clean["postcode"] = (
    plant_clean["postcode"].astype(str).str.strip()
)

type_map = {
    "House": "House",
    "Apartment": "Apartment",
    "Unit": "Apartment",
    "Studio": "Apartment",
    "Townhouse": "Townhouse/Villa"
}

plant_clean["primary_type"] = plant_clean["property_type"].map(type_map)

plant_clean["exclusion_reason"] = ""

def add_reason(mask, reason):
    plant_clean.loc[mask, "exclusion_reason"] += (
        plant_clean.loc[mask, "exclusion_reason"].apply(lambda x: "; " if x else "")
        + reason
    )
    
add_reason(plant_clean["lat"].isna() | plant_clean["lon"].isna(), "invalid_coordinate")
add_reason(plant_clean["listing_id"].astype(str) == "201885", "non_residential_property")
add_reason(plant_clean["listing_id"].astype(str) == "164784", "non_residential_storage")
add_reason(plant_clean["property_type"].eq("Other"), "unmapped_property_type")

print(plant_clean.loc[plant_clean
                        ["exclusion_reason"] != "", "exclusion_reason"].value_counts())

exclusion_reason
unmapped_property_type      28
invalid_coordinate           1
non_residential_property     1
non_residential_storage      1
Name: count, dtype: int64


In [4]:
plant_validation = plant_clean[plant_clean["exclusion_reason"] == ""].copy()

required_clean_cols = [
    "weekly_rent",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "lat",
    "lon",
    "primary_type"
]

assert plant_validation[required_clean_cols].notna().all().all()
assert plant_validation["listing_id"].is_unique
assert plant_validation["weekly_rent"].gt(0).all()

print("Rows retained:", len(plant_validation))
print("Rows excluded:", len(plant_clean) - len(plant_validation))

print("\nMapped property types:")
print(plant_validation["primary_type"].value_counts())

print("\nWeekly rent summary:")
print(plant_validation["weekly_rent"].describe())

Rows retained: 720
Rows excluded: 31

Mapped property types:
primary_type
House              434
Apartment          178
Townhouse/Villa    108
Name: count, dtype: int64

Weekly rent summary:
count     720.000000
mean      607.694444
std       178.045027
min       180.000000
25%       500.000000
50%       580.000000
75%       675.000000
max      2200.000000
Name: weekly_rent, dtype: float64


## 3. Attach SA2-level features
Retained listings are spatially matched to 2021 SA2 boundaries. The corresponding demographic, population and infrastructure features are then merged using the same SA2 feature table as the training pipeline.

In [5]:
sa2 = gpd.read_file("../data/raw/sa2_features.gpkg")

print("SA2 regions:", len(sa2))
print("CRS:", sa2.crs)

SA2 regions: 524
CRS: EPSG:7844


In [6]:
# Convert Barry Plant listings to GeoDataFrame

plant_gdf = gpd.GeoDataFrame(
    plant_validation.copy(),
    geometry=gpd.points_from_xy(
        plant_validation["lon"],
        plant_validation["lat"]
    ),
    crs="EPSG:4326"
)

# Match SA2 boundary CRS
plant_gdf = plant_gdf.to_crs(sa2.crs)

print("Plant CRS:", plant_gdf.crs)
print("SA2 CRS:", sa2.crs)
print("Rows:", len(plant_gdf))

Plant CRS: EPSG:7844
SA2 CRS: EPSG:7844
Rows: 720


In [7]:
# Spatial join: property point within SA2 polygon

plant_sa2 = gpd.sjoin(
    plant_gdf,
    sa2[["sa2_code_2021", "sa2_name_2021", "geometry"]],
    how="left",
    predicate="within"
)

print("Rows after spatial join:", len(plant_sa2))
print("Unmatched SA2:", plant_sa2["sa2_code_2021"].isna().sum())
print("Duplicate listing IDs:", plant_sa2["listing_id"].duplicated().sum())

Rows after spatial join: 720
Unmatched SA2: 1
Duplicate listing IDs: 0


In [8]:
unmatched = plant_sa2[
    plant_sa2["sa2_code_2021"].isna()
]

display(unmatched[
    ["listing_id", "address", "lat", "lon"]
])

,listing_id,address,lat,lon
233,209175,4/8 West Road,-34.170958,142.179477


In [9]:
plant_sa2.loc[plant_sa2["listing_id"].astype(str) == "209175", 
            "exclusion_reason"] = "outside_study_area"

plant_validation_sa2 = plant_sa2[plant_sa2["exclusion_reason"] == ""].copy()

print("Rows retained:", len(plant_validation_sa2))
print("Missing SA2:", plant_validation_sa2["sa2_code_2021"].isna().sum())

Rows retained: 719
Missing SA2: 0


In [10]:
sa2_features = pd.read_parquet("../data/curated/sa2_features.parquet")

plant_validation_sa2["sa2_code_2021"] = (plant_validation_sa2["sa2_code_2021"].astype("string"))

sa2_features["sa2_code_2021"] = (sa2_features["sa2_code_2021"].astype("string"))

plant_validation_ext = plant_validation_sa2.merge(
    sa2_features,
    on="sa2_code_2021",
    how="left",
    validate="many_to_one",
    suffixes=("", "_sa2"))

assert (plant_validation_ext["population_2026"].notna().all())

print("Rows after SA2 feature merge:", len(plant_validation_ext))

Rows after SA2 feature merge: 719


## 4. Prepare spatial reference datasets

The same station, school, park and shopping-centre definitions used in the training pipeline are reused to construct property-level spatial features.

In [11]:
load_dotenv()

ors_key = os.getenv("ORS_API_KEY")
assert ors_key is not None, "ORS_API_KEY not found"

print("ORS key loaded:", True)

ORS key loaded: True


### Train stations

Train station points were generated from the PTV GTFS feeds using `build_train_stations.py`, following the same preprocessing logic as the training pipeline.

In [12]:
stations = pd.read_parquet("../data/curated/vic_train_stations.parquet")

### Schools

School coordinates are standardised from the DataVic school-location dataset. Records without usable coordinates are removed.

In [13]:
schools_raw = pd.read_csv(
"../data/raw/dv378_DataVic-SchoolLocations-2024.csv",
    encoding="cp1252"
)

schools = (
    schools_raw
    .rename(columns={
        "X": "longitude",
        "Y": "latitude",
        "School_Name": "school_name"
    })[
        [
            "School_No",
            "school_name",
            "School_Type",
            "Education_Sector",
            "longitude",
            "latitude"
        ]
    ]
    .dropna(subset=["longitude", "latitude"])
    .copy()
)

schools.to_parquet("../data/curated/vic_school_points.parquet",
                    index=False)

print("School points:", len(schools))

School points: 2293


### Parks and shopping centres

OpenStreetMap park and shopping-centre records are converted to representative points. Shopping centres are restricted to the Victorian study area using the SA2 boundaries.

In [ ]:
PARKS_JSON = "../data/landing/osm/osm_parks_vic.json"

with open(PARKS_JSON, "r", encoding="utf-8") as f:
    parks_json = json.load(f)
    
len(parks_json["elements"])
parks_rows = []

for el in parks_json["elements"]:
    tags = el.get("tags", {})
    if el["type"] == "node":
        lat = el.get("lat")
        lon = el.get("lon")
    else:
        centre = el.get("center", {})
        lat = centre.get("lat")
        lon = centre.get("lon")
        
    if lat is None or lon is None:
        continue
    
    parks_rows.append({
        "osm_id": str(el["id"]),
        "name": tags.get("name"),
        "latitude": lat,
        "longitude": lon
    })
    
parks = pd.DataFrame(parks_rows)
    
print("Park points:", len(parks))
print("Named parks:", parks["name"].notna().sum())
print("Missing coordinates:", parks[["latitude", "longitude"]].isna().any(axis=1).sum())

parks.to_parquet("../data/curated/osm_parks_vic.parquet",index=False)

Park points: 17005
Named parks: 8042
Missing coordinates: 0


In [ ]:
MALLS_JSON = "../data/landing/osm/osm_malls_vic.json"

with open(MALLS_JSON, "r", encoding="utf-8") as f:
    malls_json = json.load(f)
    
mall_rows = []

for el in malls_json["elements"]:
    tags = el.get("tags", {})
    if el["type"] == "node":
        lat = el.get("lat")
        lon = el.get("lon")
    else:
        centre = el.get("center", {})
        lat = centre.get("lat")
        lon = centre.get("lon")
        
    if lat is None or lon is None:
        continue
    
    mall_rows.append({
        "osm_id": str(el["id"]),
        "name": tags.get("name"),
        "latitude": lat,
        "longitude": lon
    })
    
malls = pd.DataFrame(mall_rows)
    
print("Mall points:", len(malls))
print("Missing coordinates:", malls[["latitude", "longitude"]].isna().any(axis=1).sum())
malls_gdf = gpd.GeoDataFrame(
    malls,
    geometry=gpd.points_from_xy(malls["longitude"], malls["latitude"]),
    crs="EPSG:4326"
)

malls_gdf = malls_gdf.to_crs(sa2.crs)

malls_vic = gpd.sjoin(
    malls_gdf,
    sa2[["geometry"]],
    how="inner",
    predicate="within"
)

malls_vic = (malls_vic.drop(columns=["index_right"])
                        .drop_duplicates(subset="osm_id")
                        .reset_index(drop=True))

print("Raw mall points:", len(malls))
print("Victoria mall points", len(malls_vic))

malls_vic.to_parquet("../data/curated/osm_malls_vic.parquet", index=False)

Mall points: 433
Missing coordinates: 0
Raw mall points: 433
Victoria mall points 383


## 5. Construct property-level spatial features

Facility locations and Barry Plant properties are projected to EPSG:7855 so that Euclidean distance and radius counts are measured in metres, matching the training pipeline.

In [17]:
def to_metric_gdf(df, lon_col, lat_col):
    return gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
                            crs="EPSG:4326").to_crs("EPSG:7855")
    
plant_points = to_metric_gdf(plant_validation_ext, "lon", "lat")

station_points = to_metric_gdf(stations, "longitude", "latitude")

school_points = to_metric_gdf(schools, "longitude", "latitude")

park_points = to_metric_gdf(parks, "longitude", "latitude")

mall_points = malls_vic.to_crs("EPSG:7855")

print(
    len(plant_points),
    len(station_points),
    len(school_points),
    len(park_points),
    len(mall_points)
)

719 320 2293 17005 383


In [18]:
def get_xy(gdf):
    return np.column_stack([
        gdf.geometry.x.to_numpy(),
        gdf.geometry.y.to_numpy()
    ])
    
def nearest_distance_km(source, target):
    tree = BallTree(get_xy(target), metric="euclidean")
    dist, _ = tree.query(get_xy(source), k=1)
    return dist[:, 0] / 1000

def count_within(source, target, radius_m):
    tree = BallTree(get_xy(target), metric="euclidean")
    return tree.query_radius(get_xy(source), r=radius_m, count_only=True)

In [19]:
plant_spatial = pd.DataFrame({"property_id": plant_validation_ext["listing_id"].astype(str).to_numpy()})

# Station
plant_spatial["straight_km_to_nearest_station"] = nearest_distance_km(plant_points, station_points)
plant_spatial["n_stations_within_1km"] = count_within(plant_points, station_points, 1000) # type: ignore
plant_spatial["n_stations_within_2km"] = count_within(plant_points, station_points, 2000) # type: ignore

# School
plant_spatial["straight_km_to_nearest_school"] = nearest_distance_km(plant_points, school_points)
plant_spatial["n_schools_within_1km"] = count_within(plant_points, school_points, 1000) # pyright: ignore[reportArgumentType, reportCallIssue]
plant_spatial["n_schools_within_2km"] = count_within(plant_points, school_points, 2000) # type: ignore

# Park
plant_spatial["straight_km_to_nearest_park"] = nearest_distance_km(plant_points, park_points)
plant_spatial["n_parks_within_500m"] = count_within(plant_points, park_points, 500) # pyright: ignore[reportCallIssue, reportArgumentType]
plant_spatial["n_parks_within_1km"] = count_within(plant_points, park_points, 1000) # pyright: ignore[reportArgumentType, reportCallIssue]

#Named park
named_park_points = park_points[park_points["name"].notna()].copy()
plant_spatial["straight_km_to_nearest_named_park"] = nearest_distance_km(plant_points, named_park_points)

# Mall
plant_spatial["straight_km_to_nearest_mall"] = nearest_distance_km(plant_points, mall_points)

In [20]:
print("Shape:", plant_spatial.shape)

assert plant_spatial["property_id"].notna().all().all()

print("\nDuplicate property_id:", plant_spatial["property_id"].duplicated().sum())

display(plant_spatial.describe().T)

Shape: (719, 12)

Duplicate property_id: 0


,count,mean,std,min,25%,50%,75%,max
straight_km_to_nearest_station,719.0,5.733821,22.758523,0.036459,0.879275,1.626543,3.160978,188.431023
n_stations_within_1km,719.0,0.365786,0.620897,0.000000,0.000000,0.000000,1.000000,4.000000
n_stations_within_2km,719.0,1.301808,1.627372,0.000000,0.000000,1.000000,2.000000,10.000000
straight_km_to_nearest_school,719.0,0.704831,0.534315,0.040306,0.391999,0.595206,0.853514,5.012791
n_schools_within_1km,719.0,1.940195,1.446554,0.000000,1.000000,2.000000,3.000000,6.000000
n_schools_within_2km,719.0,7.069541,3.866599,0.000000,4.000000,7.000000,10.000000,21.000000
straight_km_to_nearest_park,719.0,0.264095,0.340206,0.017827,0.145470,0.223575,0.330108,8.125579
n_parks_within_500m,719.0,3.766342,3.119876,0.000000,2.000000,3.000000,5.000000,38.000000
n_parks_within_1km,719.0,13.664812,8.272872,0.000000,8.000000,13.000000,17.000000,78.000000
straight_km_to_nearest_named_park,719.0,0.402088,0.512566,0.020802,0.191893,0.311986,0.458862,8.184325


### Estimate route distance to the nearest station

Recover the original detour factor from saved training features and apply it to the straight-line distances. This is an estimate to the straight-line nearest station, not a separate routed nearest-station search.

In [21]:
# Recover the full-precision factor used for the training properties.
# Reuse it for external validation instead of recalibrating on Barry Plant.
training_geo = pd.read_parquet(
    "../data/curated/property_geo_features.parquet",
    columns=[
        "straight_km_to_nearest_station",
        "est_route_km_to_nearest_station",
    ],
)

valid_detour = (
    training_geo["straight_km_to_nearest_station"].gt(0)
    & np.isfinite(training_geo["straight_km_to_nearest_station"])
    & np.isfinite(training_geo["est_route_km_to_nearest_station"])
)
detour_ratios = (
    training_geo.loc[valid_detour, "est_route_km_to_nearest_station"]
    / training_geo.loc[valid_detour, "straight_km_to_nearest_station"]
)
assert not detour_ratios.empty, "No valid training rows to recover the detour factor"
DETOUR_INDEX = float(detour_ratios.median())
assert 1.0 <= DETOUR_INDEX <= 3.0, "Detour factor is outside the original calibration bounds"
assert np.allclose(detour_ratios, DETOUR_INDEX, rtol=1e-10, atol=1e-12), \
    "Training features do not use a single consistent detour factor"

print("Training rows used:", len(detour_ratios))
print("Recovered detour factor:", repr(DETOUR_INDEX))


Training rows used: 11945
Recovered detour factor: 1.5368313765358042


In [22]:
plant_spatial["est_route_km_to_nearest_station"] = (
    plant_spatial["straight_km_to_nearest_station"] * DETOUR_INDEX
)

assert np.isfinite(plant_spatial["est_route_km_to_nearest_station"]).all()
assert (
    plant_spatial["est_route_km_to_nearest_station"]
    >= plant_spatial["straight_km_to_nearest_station"]
).all()

print("Properties with estimated station route distance:", len(plant_spatial))

Properties with estimated station route distance: 719


### Driving route to Melbourne CBD

Driving distance and duration to Flinders Street Station are obtained using OpenRouteService, match the CBD reference point and driving profile used in the training pipeline.

In [23]:
CBD_LAT = -37.8183
CBD_LON = 144.9671

def ors_matrix_to_cbd(lons, lats, api_key):
    locations = [[lon, lat] for lon, lat in zip(lons, lats)]
    locations.append([CBD_LON, CBD_LAT])
    
    cbd_idx = len(locations) - 1
    
    payload = {
        "locations": locations,
        "sources": list(range(len(lons))),
        "destinations": [cbd_idx],
        "metrics": ["distance", "duration"],
        "units": "m"
    }
    
    headers = {"Authorization": api_key, "Content-Type": "application/json"}
    
    responses = requests.post(
        "https://api.openrouteservice.org/v2/matrix/driving-car",
        json=payload,
        headers=headers,
        timeout=120
    )
    
    responses.raise_for_status()
    result = responses.json()
    
    distances_m = np.array(result["distances"])[:, 0]
    durations_s = np.array(result["durations"])[:, 0]
    
    return distances_m / 1000, durations_s / 60

In [24]:
route_km, route_min = ors_matrix_to_cbd(
    plant_validation_ext["lon"].to_numpy(),
    plant_validation_ext["lat"].to_numpy(),
    ors_key
)

plant_spatial["route_km_to_cbd"] = route_km
plant_spatial["route_min_to_cbd"] = route_min

In [25]:
assert plant_spatial[["route_km_to_cbd", "route_min_to_cbd"]].notna().all().all()
assert (plant_spatial["route_km_to_cbd"] >= 0).all()
assert (plant_spatial["route_min_to_cbd"] >= 0).all()

display(plant_spatial[["route_km_to_cbd", "route_min_to_cbd"]].describe())

,route_km_to_cbd,route_min_to_cbd
count,719.000000,719.000000
mean,47.750961,47.968029
std,69.087291,46.652471
min,1.667840,4.909833
25%,20.241720,30.385167
50%,30.133520,38.883333
75%,50.330625,50.060417
max,555.827310,394.064500


## 6. Build the final validation feature table

Property-level spatial features are merged with the cleaned Barry Plant listings and SA2-level attributes. The resulting dataset is reserved for external validation.

In [26]:
plant_validation_ext["listing_id"] = (plant_validation_ext["listing_id"].astype(str))

plant_validation_features = plant_validation_ext.merge(plant_spatial,
                                                        left_on="listing_id",
                                                        right_on="property_id",
                                                        how="left",
                                                        validate="one_to_one")

In [27]:
required_features = [
    "bedrooms", "bathrooms", "carspaces",
    "route_km_to_cbd",
    "est_route_km_to_nearest_station",
    "straight_km_to_nearest_school",
    "straight_km_to_nearest_park",
    "straight_km_to_nearest_mall",
    "n_stations_within_1km",
    "n_schools_within_1km",
    "n_parks_within_500m",
    "income_median",
    "population_growth_5y",
    "population_density_2026",
    "primary_type", "suburb",
]

missing_cols = set(required_features + ["weekly_rent"]) - set(
    plant_validation_features.columns
)

assert not missing_cols, f"Missing columns: {missing_cols}"
assert plant_validation_features["listing_id"].is_unique
assert plant_validation_features["weekly_rent"].notna().all()
assert plant_validation_features["weekly_rent"].gt(0).all()

assert (plant_validation_features[plant_spatial.columns.drop("property_id")]
        .notna().all().all())

print("Final shape:", plant_validation_features.shape)
print("Listings:", len(plant_validation_features))
print("Required model features complete:", True)

plant_validation_features.to_parquet(
    "../data/curated/plant_validation_features.parquet",
    index=False,
)

print("Saved to:", "../data/curated/plant_validation_features.parquet")

Final shape: (719, 71)
Listings: 719
Required model features complete: True
Saved to: ../data/curated/plant_validation_features.parquet


## Output

The final validation dataset contains **719 Barry Plant rental listings** with aligned property, SA2 and spatial features.

The dataset is saved as:

`data/curated/plant_validation_features.parquet`

It is used only for external validation of the previously trained rental-price model and is not used for retraining, feature selection or hyperparameter tuning